# Learning knowledge distillation by building a tiny query router

**Student Notes.** I want to see whether a small network can copy a larger one on a controlled four-route task. The vectors here are synthetic; success here is a learning exercise, not evidence that real user queries are safe to route.

## 1. Getting started
I first check the runtime. MPS helps training on an Apple GPU; I will still time inference on CPU.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()))
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader
from src.dataset import AgentQueryDataset, ROUTES
from src.models import TeacherNet, StudentNet
from src.losses import DistillationLoss
from src.benchmark import count_parameters, measure_latency, evaluate_accuracy
print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Training device:", device)

## 2. The data shape
One query becomes 384 numbers. Four clusters stand in for related semantic intents. They are generated vectors, not encoded sentences.

In [ ]:
data = AgentQueryDataset(samples_per_class=500)
x, label = data[0]
print("Embedding shape:", x.shape, "route:", label.item(), ROUTES[label])
print("First twelve coordinates:", x[:12].round(decimals=2))
print("Classes:", list(enumerate(ROUTES)))

## 3. My first layer
A linear layer maps 384 coordinates to four arbitrary scores. They are **logits**, not probabilities: they need not sum to one or even be positive.

In [ ]:
layer = nn.Linear(384, 4)
logits = layer(torch.randn(1, 384))
print("Raw logits:", logits)
print("Probabilities:", logits.softmax(dim=1), "sum:", logits.softmax(dim=1).sum().item())

## 4. Temperature: where the dark knowledge appears
**Wait, Why Does This Work?** A hard label only says which route won. A teacher can also tell us which other routes were plausible. Dividing logits by temperature before softmax makes those smaller probabilities visible.

In [ ]:
example = torch.tensor([[8.0, 3.0, 1.0, -2.0]])
for temperature in (1, 4):
    print(f"T={temperature}:", F.softmax(example / temperature, dim=1).round(decimals=4))

**Hypothesis vs. Observation.** I expected the top class to stay on top while its probability fell. The output should confirm that: the ranking remains, but the non-winning classes carry more signal. A high temperature does not manufacture truth; it only exposes the teacher’s relative scores.

## 5. Teacher and student
The teacher has four linear layers and dropout during training. The student has just two linear layers. I want to quantify the compression before asking whether it learns.

In [ ]:
teacher, student = TeacherNet(), StudentNet()
print("Teacher parameters:", count_parameters(teacher))
print("Student parameters:", count_parameters(student))
print("Reduction:", f"{1-count_parameters(student)/count_parameters(teacher):.1%}")

## 6. Blended distillation loss
The hard-label term keeps the task anchored. The KL term asks the student to match the teacher’s softened distribution.

**Student Notes.** Softmax derivatives shrink roughly as $1/T$, and the KL gradient with respect to the unscaled logits gains another $1/T$. Multiplying the KL term by $T^2$ approximately restores its gradient scale, so changing temperature does not silently weaken this term. This is a gradient correction, not a promise that every temperature works equally well. The teacher logits are detached because the teacher is the fixed target in this step.

In [ ]:
criterion = DistillationLoss(temperature=4)
dummy_x = torch.randn(8, 384)
dummy_y = torch.randint(0, 4, (8,))
with torch.no_grad():
    teacher_logits = teacher(dummy_x)
print("Blended loss:", criterion(student(dummy_x), teacher_logits, dummy_y).item())

## 7. Training
I train all three models so the baseline tells me whether distillation actually helped. The held-out examples use fresh noise around the same synthetic centroids.

In [ ]:
from train import train_model
torch.manual_seed(42)
torch.set_num_threads(1)
train_loader = DataLoader(AgentQueryDataset(500, seed=42), batch_size=64, shuffle=True)
test_loader = DataLoader(AgentQueryDataset(100, seed=43), batch_size=128)
teacher = train_model(TeacherNet(), train_loader, 8, device).eval()
baseline = train_model(StudentNet(), train_loader, 8, device).eval()
distilled = train_model(StudentNet(), train_loader, 8, device, teacher).eval()
for name, model in (("Teacher", teacher), ("Baseline", baseline), ("Distilled", distilled)):
    print(name, f"{evaluate_accuracy(model, test_loader, device):.1%}")

## 8. CPU profiling
I time 1,000 single-query forward passes after 100 warm-up passes. p50 is the middle sample; p95 captures slower runs. This excludes embedding generation and any application overhead.

In [ ]:
for name, model in (("Teacher", teacher), ("Baseline", baseline), ("Distilled", distilled)):
    p50, p95 = measure_latency(model, passes=1000)
    print(f"{name:10s} p50={p50:.4f} ms  p95={p95:.4f} ms")

**Hypothesis vs. Observation.** The student should have lower latency and far fewer weights. If both students reach the same accuracy, I cannot claim distillation improved this easy synthetic task. To test routing fidelity, the next experiment needs real query embeddings, ambiguous prompts, and a separate sensitive-action error analysis.